In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
df = pd.read_csv("../data/DataCoSupplyChainDataset.csv", encoding='latin1')
print(df.shape)

(180519, 53)


In [2]:
df.isnull().sum().sort_values(ascending=False)

Product Description              180519
Order Zipcode                    155679
Customer Lname                        8
Customer Zipcode                      3
Type                                  0
Order Profit Per Order                0
Order Item Cardprod Id                0
Order Item Discount                   0
Order Item Discount Rate              0
Order Item Id                         0
Order Item Product Price              0
Order Item Profit Ratio               0
Order Item Quantity                   0
Sales                                 0
Order Item Total                      0
Order Region                          0
order date (DateOrders)               0
Order State                           0
Order Status                          0
Product Card Id                       0
Product Category Id                   0
Product Image                         0
Product Name                          0
Product Price                         0
Product Status                        0


In [3]:
missing_percentage = (df.isnull().sum()/len(df))*100
missing_percentage.sort_values(ascending=False)

Product Description              100.000000
Order Zipcode                     86.239676
Customer Lname                     0.004432
Customer Zipcode                   0.001662
Type                               0.000000
Order Profit Per Order             0.000000
Order Item Cardprod Id             0.000000
Order Item Discount                0.000000
Order Item Discount Rate           0.000000
Order Item Id                      0.000000
Order Item Product Price           0.000000
Order Item Profit Ratio            0.000000
Order Item Quantity                0.000000
Sales                              0.000000
Order Item Total                   0.000000
Order Region                       0.000000
order date (DateOrders)            0.000000
Order State                        0.000000
Order Status                       0.000000
Product Card Id                    0.000000
Product Category Id                0.000000
Product Image                      0.000000
Product Name                    

In [4]:
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 0


In [5]:
# Removing unnecessary columns
columns_to_drop = [
    'Customer Email',
    'Customer Password',
    'Customer Fname',
    'Customer Lname',
    'Customer Street',
    'Customer Zipcode',
    'Product Description'
]

df.drop(columns=columns_to_drop, inplace=True, errors='ignore')

print(df.shape)

(180519, 46)


In [6]:
df.isnull().sum()[df.isnull().sum() > 0]

Order Zipcode    155679
dtype: int64

In [7]:
# Converting date columns to datetime
df['order date (DateOrders)'] = pd.to_datetime(
    df['order date (DateOrders)']
)
df['shipping date (DateOrders)'] = pd.to_datetime(
    df['shipping date (DateOrders)']
)

In [8]:
# Creating shipping duration feature
df['Shipping_Duration'] = (
    df['shipping date (DateOrders)']
    - df['order date (DateOrders)']
).dt.days
df[['Shipping_Duration']].head()

,Shipping_Duration
0,3
1,5
2,4
3,3
4,2


In [9]:
# Creating order value feature
df['Order_Value'] = (
    df['Order Item Quantity']
    * df['Order Item Product Price']
)

In [10]:
# Creating profit margin feature
df['Profit_Margin'] = (
    df['Order Profit Per Order']/(df['Order Item Total'] + 1)
)

In [11]:
# Creating estimated freight cost feature
df['Estimated_Freight_Cost'] = (
    df['Order_Value'] * 0.08 + df['Shipping_Duration'] * 2
)
df[['Estimated_Freight_Cost']].head()

,Estimated_Freight_Cost
0,32.22
1,36.22
2,34.22
3,32.22
4,30.22


In [12]:
# Creating high value order flag
threshold = df['Order_Value'].quantile(0.75)
df['High_Value_Order'] = np.where(
    df['Order_Value'] > threshold,
    1,
    0
)

In [13]:
# Creating risk flag
df['Risk_Flag'] = np.where(
    (
        (df['Late_delivery_risk'] == 1)
        |
        (df['Shipping_Duration'] > 7)
        |
        (df['High_Value_Order'] == 1)
    ),
    1,
    0
)

In [14]:
# Verifing the target variables
print(df['Estimated_Freight_Cost'].describe())

count    180519.000000
mean         23.245480
std          11.105417
min           0.799200
25%          15.995200
50%          21.993600
75%          28.000000
max         171.999199
Name: Estimated_Freight_Cost, dtype: float64


In [15]:
print(df['Risk_Flag'].value_counts())

Risk_Flag
1    118158
0     62361
Name: count, dtype: int64


In [16]:
# Checking new features
df[[
    'Shipping_Duration',
    'Order_Value',
    'Profit_Margin',
    'Estimated_Freight_Cost',
    'Risk_Flag'
]].head()

,Shipping_Duration,Order_Value,Profit_Margin,Estimated_Freight_Cost,Risk_Flag
0,3,327.75,0.289095,32.22,1
1,5,327.75,-0.797445,36.22,1
2,4,327.75,-0.797438,34.22,1
3,3,327.75,0.074752,32.22,1
4,2,327.75,0.448488,30.22,1


In [17]:
# Saving the clean dataset
df.to_csv("../data/cleaned_supply_chain_data.csv", index=False)
print("Clean dataset saved successfully!")

Clean dataset saved successfully!


In [18]:
df.shape

(180519, 52)